# Substrate pairs in each dataset

What the substrate pair is in every screen, how many reactions each pair has,
which other variables change while the pair is held fixed, and **how much of the
yield variance the pair alone explains**. Descriptive only — nothing here
changes a bundle or a split.

The motivating question: if a model's score comes mostly from recognising the
substrate pair, its ligand/condition descriptors are not doing the work. The
**pair-mean R²** below is the R² of predicting every reaction by its pair's mean
yield (in-sample, so an upper bound for a pair-only model). A trained model has
to beat it to be using anything beyond the pairing. The same quantity is given
for the ligand/catalyst alone and for each (pair, ligand) cell.

The (pair, ligand) cell table at the end of each dataset is the inventory for a
later "hold out one substrate pair *and* one ligand" split: it shows which cells
exist and how big they are.

`ahneman_doyle_dft` has exactly the `ahneman` reactions, so it is not repeated.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()          # gp_collab_hub/
sys.path.insert(0, str(ROOT))
from gpc.config import bundle_group, bundle_name, bundle_target, condition_numeric
from gpc.data import load_bundle

pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 60)


# ---- Editable: what the substrate pair is in each screen -----------------
# electrophile/nucleophile columns of reactions.csv; a fixed partner is named
# as text instead (Ahneman couples every aryl halide with p-toluidine).
PAIRS = {
    "perera":         {"electrophile": "Reactant_1_Short_Hand", "nucleophile": "Reactant_2_Name"},
    "gesmundo":       {"electrophile": "core", "nucleophile": "monomer"},
    "ahneman":        {"electrophile": "aryl_halide", "nucleophile": "p-toluidine (fixed)"},
    "reizman_summit": {"electrophile": "aryl_halide", "nucleophile": "boron_reagent"},
}
# --------------------------------------------------------------------------

In [2]:
def r2_of_means(y, keys):
    # R^2 of predicting each row by the mean of its group (= eta squared).
    fitted = y.groupby(keys).transform("mean")
    return float(1 - ((y - fitted) ** 2).sum() / ((y - y.mean()) ** 2).sum())


def load(name):
    reactions, _, _, prepared = load_bundle(ROOT / "datasets" / name / "inputs")
    spec = PAIRS[name]
    electrophile = reactions[spec["electrophile"]].astype(str)
    if spec["nucleophile"] in reactions:
        nucleophile = reactions[spec["nucleophile"]].astype(str)
    else:
        nucleophile = pd.Series(spec["nucleophile"], index=reactions.index)
    reactions = reactions.assign(electrophile=electrophile, nucleophile=nucleophile,
                                 pair=electrophile + "  +  " + nucleophile)
    return reactions, prepared


def varied(reactions, prepared, spec):
    # Every model input other than the substrate pair, with its number of levels.
    group = bundle_group(prepared)
    pair_columns = {spec["electrophile"], spec["nucleophile"], "substrate_pair"}
    fields = [group] + [c for c in prepared["data"]["common_categorical"] if c not in pair_columns]
    out = {c: f"{reactions[c].nunique()} levels" for c in fields}
    for c in condition_numeric(prepared):
        out[c] = f"continuous {reactions[c].min():g}-{reactions[c].max():g}"
    return out


summary, per_dataset = [], {}
for name, spec in PAIRS.items():
    reactions, prepared = load(name)
    y = reactions[bundle_target(prepared)].astype(float)
    group = bundle_group(prepared)
    n_e, n_n = reactions.electrophile.nunique(), reactions.nucleophile.nunique()
    sizes = reactions.pair.value_counts()
    cells = reactions.groupby(["pair", group]).size()
    summary.append({
        "dataset": bundle_name(prepared),
        "rows": len(reactions),
        "electrophiles": n_e, "nucleophiles": n_n,
        "pairs (observed / design)": f"{reactions.pair.nunique()} / {n_e * n_n}",
        "rows per pair (min-median-max)": f"{sizes.min()}-{int(sizes.median())}-{sizes.max()}",
        f"groups": f"{reactions[group].nunique()} {group}s",
        "R2 pair mean": round(r2_of_means(y, reactions.pair), 3),
        "R2 group mean": round(r2_of_means(y, reactions[group]), 3),
        "R2 (pair, group) mean": round(r2_of_means(y, [reactions.pair, reactions[group]]), 3),
        "(pair, group) cells": f"{len(cells)} of {reactions.pair.nunique() * reactions[group].nunique()}",
        "rows per cell (min-median-max)": f"{cells.min()}-{int(cells.median())}-{cells.max()}",
        "varied with pair fixed": "; ".join(f"{k} ({v})" for k, v in varied(reactions, prepared, spec).items()),
    })
    per_dataset[name] = (reactions, prepared)

summary = pd.DataFrame(summary).set_index("dataset")
display(summary.T)

dataset,Perera,Gesmundo,Ahneman,Reizman_Summit
rows,3072,1320,3955,332
electrophiles,4,12,15,3
nucleophiles,3,10,1,3
pairs (observed / design),12 / 12,115 / 120,15 / 15,4 / 9
rows per pair (min-median-max),256-256-256,2-12-12,262-264-264,59-91-91
groups,8 ligands,5 catalysts,4 ligands,6 ligands
R2 pair mean,0.536,0.619,0.426,0.375
R2 group mean,0.029,0.025,0.084,0.194
"R2 (pair, group) mean",0.673,0.88,0.556,0.625
"(pair, group) cells",96 of 96,564 of 575,60 of 60,24 of 24


**Reading the R² columns.** "pair mean" near 0 means yield is not decided by the
substrates — the other variables carry it, and a model has to learn them.
Near 1 means a model could score well by recognising the pair and little else,
which is the failure mode to guard against. "(pair, group) mean" is the ceiling
for a model that knows only the substrates and the ligand/catalyst and ignores
every remaining condition.

In [3]:
def pair_table(name):
    reactions, prepared = per_dataset[name]
    target, group = bundle_target(prepared), bundle_group(prepared)
    table = (reactions.groupby(["pair", "electrophile", "nucleophile"])
             .agg(rows=(target, "size"), groups=(group, "nunique"),
                  mean=(target, "mean"), sd=(target, "std"),
                  min=(target, "min"), max=(target, "max"),
                  share_below_5=(target, lambda s: (s < 5).mean()))
             .reset_index()
             .sort_values("mean", ascending=False, kind="stable")
             .reset_index(drop=True))
    table.insert(0, "rank", range(1, len(table) + 1))
    table[["mean", "sd", "min", "max"]] = table[["mean", "sd", "min", "max"]].round(1)
    table["share_below_5"] = (100 * table["share_below_5"]).round(0).astype(int).astype(str) + "%"
    table = table.rename(columns={"groups": f"{group}s", "share_below_5": "yield < 5%"})
    return table.drop(columns="pair")


def cell_table(name, values="count"):
    # (pair x group) inventory: row counts, or mean yield, per cell.
    reactions, prepared = per_dataset[name]
    target, group = bundle_target(prepared), bundle_group(prepared)
    how = "size" if values == "count" else "mean"
    pivot = reactions.pivot_table(index="pair", columns=group, values=target,
                                  aggfunc=how, fill_value=0 if how == "size" else np.nan)
    return pivot.round(1) if how == "mean" else pivot.astype(int)

## Perera

4 quinoline electrophiles (6-Cl, -Br, -I, -OTf) × 3 boron nucleophiles (acid, pinacol ester, trifluoroborate) — a complete 12-pair grid. Note `substrate_pair` and `Reactant_1_Short_Hand` are already one-hot inputs in every Perera section.

In [4]:
display(pair_table("perera"))

,rank,electrophile,nucleophile,rows,ligands,mean,sd,min,max,yield < 5%
0,1,"1d, 6-I-Q","2b, Boronic Ester",256,8,80.2,14.6,8.7,100.0,0%
1,2,"1d, 6-I-Q","2a, Boronic Acid",256,8,79.3,15.7,8.8,98.7,0%
2,3,"1b, 6-Br-Q","2a, Boronic Acid",256,8,67.8,23.6,9.5,97.9,0%
3,4,"1c, 6-OTf-Q","2a, Boronic Acid",256,8,64.6,24.0,8.7,98.5,0%
4,5,"1c, 6-OTf-Q","2b, Boronic Ester",256,8,60.8,23.2,10.3,98.6,0%
5,6,"1b, 6-Br-Q","2b, Boronic Ester",256,8,59.3,21.1,0.0,96.8,0%
6,7,"1d, 6-I-Q","2c, Trifluoroborate",256,8,33.3,21.6,4.7,84.8,0%
7,8,"1a, 6-Cl-Q","2a, Boronic Acid",256,8,29.4,28.4,1.0,93.7,23%
8,9,"1b, 6-Br-Q","2c, Trifluoroborate",256,8,26.3,18.3,0.0,74.8,0%
9,10,"1a, 6-Cl-Q","2b, Boronic Ester",256,8,26.3,23.0,0.0,92.5,2%


In [5]:
display(cell_table("perera"))
display(cell_table("perera", "mean"))

ligand,AmPhos,CataCXium A,P(Cy)3,P(Ph)3,P(o-Tol)3,P(tBu)3,SPhos,XPhos
pair,,,,,,,,
"1a, 6-Cl-Q + 2a, Boronic Acid",32,32,32,32,32,32,32,32
"1a, 6-Cl-Q + 2b, Boronic Ester",32,32,32,32,32,32,32,32
"1a, 6-Cl-Q + 2c, Trifluoroborate",32,32,32,32,32,32,32,32
"1b, 6-Br-Q + 2a, Boronic Acid",32,32,32,32,32,32,32,32
"1b, 6-Br-Q + 2b, Boronic Ester",32,32,32,32,32,32,32,32
"1b, 6-Br-Q + 2c, Trifluoroborate",32,32,32,32,32,32,32,32
"1c, 6-OTf-Q + 2a, Boronic Acid",32,32,32,32,32,32,32,32
"1c, 6-OTf-Q + 2b, Boronic Ester",32,32,32,32,32,32,32,32
"1c, 6-OTf-Q + 2c, Trifluoroborate",32,32,32,32,32,32,32,32


ligand,AmPhos,CataCXium A,P(Cy)3,P(Ph)3,P(o-Tol)3,P(tBu)3,SPhos,XPhos
pair,,,,,,,,
"1a, 6-Cl-Q + 2a, Boronic Acid",19.8,26.8,27.3,9.6,7.0,23.3,58.0,63.1
"1a, 6-Cl-Q + 2b, Boronic Ester",12.8,25.9,22.7,14.8,7.2,19.5,51.9,55.5
"1a, 6-Cl-Q + 2c, Trifluoroborate",16.3,19.9,18.6,20.4,16.9,17.1,21.5,23.1
"1b, 6-Br-Q + 2a, Boronic Acid",75.2,84.3,66.9,82.7,47.8,73.5,60.8,51.1
"1b, 6-Br-Q + 2b, Boronic Ester",69.9,70.3,54.5,69.3,47.5,61.8,55.8,45.3
"1b, 6-Br-Q + 2c, Trifluoroborate",28.6,32.6,17.9,35.4,29.4,24.3,21.0,21.6
"1c, 6-OTf-Q + 2a, Boronic Acid",58.0,60.9,77.0,86.4,49.1,43.3,71.6,70.3
"1c, 6-OTf-Q + 2b, Boronic Ester",52.9,53.9,72.1,87.6,46.2,39.1,69.0,65.9
"1c, 6-OTf-Q + 2c, Trifluoroborate",19.8,23.9,34.8,39.9,20.6,16.5,21.1,25.4


## Gesmundo

12 cores × 10 monomers, i.e. 120 pairs of only ~11 rows each. With so few rows per pair, a pair-mean model has little to stand on — but so does any split that holds pairs out. Cores and monomers are identifiers in the source (`Core_xx`, MFCD numbers).

In [6]:
display(pair_table("gesmundo"))

,rank,electrophile,nucleophile,rows,catalysts,mean,sd,min,max,yield < 5%
0,1,Core_03,MFCD03789252,12,5,98.0,5.4,81.7,100.0,0%
1,2,Core_05,MFCD18383786,12,5,96.7,5.1,83.6,100.0,0%
2,3,Core_05,MFCD03788418,10,5,96.3,3.6,89.9,100.0,0%
3,4,Core_06,MFCD03789252,12,5,93.0,16.1,54.6,100.0,0%
4,5,Core_03,MFCD18383786,12,5,92.7,22.8,20.4,100.0,0%
...,...,...,...,...,...,...,...,...,...,...
110,111,Core_11,MFCD19439767,7,4,0.0,0.0,0.0,0.0,100%
111,112,Core_12,MFCD00674177,12,5,0.0,0.0,0.0,0.0,100%
112,113,Core_12,MFCD03788418,12,5,0.0,0.0,0.0,0.0,100%
113,114,Core_12,MFCD18383786,12,5,0.0,0.0,0.0,0.0,100%


In [7]:
cells = cell_table("gesmundo")
print(f"{(cells > 0).sum().sum()} of {cells.size} (pair, catalyst) cells occupied; "
      f"rows per occupied cell: {cells[cells > 0].stack().describe()[['min', '50%', 'max']].to_dict()}")
display(cells.sum().rename("rows").to_frame().T)

564 of 575 (pair, catalyst) cells occupied; rows per occupied cell: {'min': 1.0, '50%': 2.0, 'max': 4.0}


catalyst,Aphos G3,RuPhos G3,Xphos G3,tBu3P G2,tBuXphos G3
rows,441,221,217,226,215


## Ahneman

Every reaction couples p-toluidine, so the substrate pair is the aryl halide alone: 15 of them (3 halogens × 5 arenes), each with ~264 reactions spanning 4 ligands × 3 bases × 22 additives.

In [8]:
display(pair_table("ahneman"))

,rank,electrophile,nucleophile,rows,ligands,mean,sd,min,max,yield < 5%
0,1,2-iodopyridine,p-toluidine (fixed),264,4,58.6,26.8,0.0,100.0,2%
1,2,1-ethyl-4-iodobenzene,p-toluidine (fixed),264,4,52.6,24.2,0.0,85.9,3%
2,3,2-bromopyridine,p-toluidine (fixed),264,4,52.4,26.5,0.0,97.3,4%
3,4,3-iodopyridine,p-toluidine (fixed),263,4,51.3,29.0,0.0,98.3,6%
4,5,1-bromo-4-ethylbenzene,p-toluidine (fixed),264,4,43.5,24.7,0.0,86.6,6%
5,6,2-chloropyridine,p-toluidine (fixed),264,4,43.5,26.9,0.0,91.4,6%
6,7,3-bromopyridine,p-toluidine (fixed),264,4,43.0,28.8,0.0,98.7,10%
7,8,1-iodo-4-(trifluoromethyl)benzene,p-toluidine (fixed),263,4,33.7,13.5,0.0,55.6,5%
8,9,1-iodo-4-methoxybenzene,p-toluidine (fixed),264,4,32.5,17.9,0.0,68.2,9%
9,10,1-bromo-4-(trifluoromethyl)benzene,p-toluidine (fixed),263,4,26.9,13.6,0.0,54.2,8%


In [9]:
display(cell_table("ahneman"))
display(cell_table("ahneman", "mean"))

ligand,AdBrettPhos,XPhos,t-BuBrettPhos,t-BuXPhos
pair,,,,
1-bromo-4-(trifluoromethyl)benzene + p-toluidine (fixed),66,66,66,65
1-bromo-4-ethylbenzene + p-toluidine (fixed),66,66,66,66
1-bromo-4-methoxybenzene + p-toluidine (fixed),66,66,66,66
1-chloro-4-(trifluoromethyl)benzene + p-toluidine (fixed),66,66,66,66
1-chloro-4-ethylbenzene + p-toluidine (fixed),66,66,66,64
1-chloro-4-methoxybenzene + p-toluidine (fixed),66,66,66,66
1-ethyl-4-iodobenzene + p-toluidine (fixed),66,66,66,66
1-iodo-4-(trifluoromethyl)benzene + p-toluidine (fixed),66,66,66,65
1-iodo-4-methoxybenzene + p-toluidine (fixed),66,66,66,66


ligand,AdBrettPhos,XPhos,t-BuBrettPhos,t-BuXPhos
pair,,,,
1-bromo-4-(trifluoromethyl)benzene + p-toluidine (fixed),24.8,21.8,28.3,32.8
1-bromo-4-ethylbenzene + p-toluidine (fixed),44.3,25.6,49.8,54.2
1-bromo-4-methoxybenzene + p-toluidine (fixed),28.2,11.1,30.0,34.2
1-chloro-4-(trifluoromethyl)benzene + p-toluidine (fixed),7.3,12.1,10.4,20.2
1-chloro-4-ethylbenzene + p-toluidine (fixed),1.4,3.7,5.5,4.8
1-chloro-4-methoxybenzene + p-toluidine (fixed),1.8,1.9,1.7,2.4
1-ethyl-4-iodobenzene + p-toluidine (fixed),58.8,33.3,60.9,57.3
1-iodo-4-(trifluoromethyl)benzene + p-toluidine (fixed),34.1,30.4,34.6,35.9
1-iodo-4-methoxybenzene + p-toluidine (fixed),39.0,15.5,37.5,38.1


## Reizman_Summit

4 pairs, which with only 3 electrophiles and 3 nucleophiles is a partial grid. Within a pair, the ligand, scaffold and three continuous conditions vary.

In [10]:
display(pair_table("reizman_summit"))

,rank,electrophile,nucleophile,rows,ligands,mean,sd,min,max,yield < 5%
0,1,2-chloropyridine,N-Boc-pyrrole-2-boronic acid,91,6,68.1,29.8,0.1,99.8,10%
1,2,3-chloropyridine,benzofuran-2-boronic acid,91,6,58.3,35.1,0.2,94.1,20%
2,3,3-bromoquinoline,"3,5-dimethylisoxazole-4-boronic acid pinacol ester",59,6,31.8,30.0,0.6,85.4,31%
3,4,3-chloropyridine,"3,5-dimethylisoxazole-4-boronic acid pinacol ester",91,6,15.1,13.8,0.1,47.6,30%


In [11]:
display(cell_table("reizman_summit"))
display(cell_table("reizman_summit", "mean"))

ligand,PCy3,PPh3,PtBu3,RuPhos,SPhos,XPhos
pair,,,,,,
2-chloropyridine + N-Boc-pyrrole-2-boronic acid,7,5,5,16,24,34
"3-bromoquinoline + 3,5-dimethylisoxazole-4-boronic acid pinacol ester",8,5,11,11,5,19
"3-chloropyridine + 3,5-dimethylisoxazole-4-boronic acid pinacol ester",24,5,9,15,10,28
3-chloropyridine + benzofuran-2-boronic acid,6,4,7,24,21,29


ligand,PCy3,PPh3,PtBu3,RuPhos,SPhos,XPhos
pair,,,,,,
2-chloropyridine + N-Boc-pyrrole-2-boronic acid,35.0,18.9,15.9,77.2,81.2,76.3
"3-bromoquinoline + 3,5-dimethylisoxazole-4-boronic acid pinacol ester",21.2,3.5,47.2,40.9,22.3,32.2
"3-chloropyridine + 3,5-dimethylisoxazole-4-boronic acid pinacol ester",27.5,0.1,7.6,12.8,10.3,12.5
3-chloropyridine + benzofuran-2-boronic acid,1.7,0.2,27.0,66.8,69.0,70.7
